# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates loading, overview, extraction, processing, and visualization of the FAIR^2 dataset using the `mlcroissant` library. All entities, such as record sets and fields, are referenced by their `@id` values for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"Dataset Name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Identifier: {metadata['identifier']}")
print(f"Published: {metadata['datePublished']}")
print("\nKeywords:", ', '.join(metadata['keywords']))

## 2. Data Overview
Review available record sets, fields, and their IDs.

The FAIR^2 dataset is comprised of one or more record sets as defined in the Croissant schema. For each record set, you can inspect its fields and column structure. All references use the `@id` identifier.

In [ ]:
# Obtain all record set @id's
record_sets = [rset['@id'] for rset in dataset.metadata.to_json().get('recordSet', [])]

if not record_sets:
    print("No record sets discovered in metadata under 'recordSet'. Attempting fallback discovery...")
    # Fallback: Try to extract from the top-level Croissant dataset schema
    meta_struct = dataset.metadata.to_json()
    for k, v in meta_struct.items():
        if isinstance(v, dict) and v.get('@type') == 'RecordSet':
            record_sets.append(v['@id'])
        if isinstance(v, list):
            for el in v:
                if isinstance(el, dict) and el.get('@type') == 'RecordSet':
                    record_sets.append(el['@id'])

if record_sets:
    print("Found RecordSet(s) by @id:")
    for rid in record_sets:
        print("  -", rid)
else:
    print("No RecordSet @id found in dataset.")

# For each record set, inspect fields
for rset_id in record_sets:
    rset_obj = dataset._get_entity(rset_id)
    print(f"\nRecordSet @id: {rset_id}")
    fields = rset_obj.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("Fields (@id):")
    for field in fields:
        fid = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
        print(f"  - {fid}")

    columns = rset_obj.get('column', [])
    if columns:
        if isinstance(columns, dict):
            columns = [columns]
        print("Columns (@id):")
        for col in columns:
            print(f"  - {col['@id']}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Each record set can be loaded as a pandas DataFrame; columns will represent their fields as referenced by their `@id`. Here we attempt to load all available record sets.

In [ ]:
# Prepare for extraction
dataframes = {}

for record_set_id in record_sets:
    print(f"\nExtracting records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Fields: {list(df.columns)}")
        print("Sample records:")
        display(df.head())
    else:
        print("No records available for this RecordSet.")

# Select one record set to use for further analysis
chosen_record_set = record_sets[0] if record_sets else None
if chosen_record_set:
    print(f"\nColumns for RecordSet @id {chosen_record_set}: {dataframes[chosen_record_set].columns.tolist()}")
    dataframes[chosen_record_set].head()
else:
    print("No record set found for further steps.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We demonstrate filtering, normalization, and grouping using fields referenced only by their `@id`. Please adjust fields and values based on your dataset content.

In [ ]:
# Choose a numeric field

# Identify likely numeric fields by inspecting DataFrame dtypes
if chosen_record_set is not None:
    df = dataframes[chosen_record_set]
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    else:
        print("No numeric fields detected.")
        numeric_field_id = None

    # Filter by a threshold if numeric field exists
    threshold = 10
    if numeric_field_id is not None:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold} (referenced by @id):")
        display(filtered_df.head())

        # Normalize numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} column for filtered DataFrame:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by another field (categorical) if available
        non_numeric_candidates = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
        group_field_id = non_numeric_candidates[0] if non_numeric_candidates else None
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id} (@id references):")
            display(grouped_df.head())
    else:
        print("No numeric fields detected; skipping filtering and normalization.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using fields referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_record_set is not None and numeric_field_id is not None:
    # Distribution plot
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[chosen_record_set][numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Scatter plot to a categorical field if available
    if group_field_id is not None:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[chosen_record_set])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Insufficient numeric or categorical fields for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset using `mlcroissant`, referencing all data entities by their `@id`. We demonstrated loading of record sets, overviewing their fields, filtering and normalizing data, grouping by categories, and performing basic visualizations. This approach ensures traceability and reproducibility throughout the data workflow.

You may further tailor analysis and visualization steps to focus on particular clinical or molecular variables relevant to the dataset's scope and research goals.